# Latent Factor Models — Exploratory Analysis

Explore the common model panel, all 19 lagged characteristics, training PCA variance,
and the completed full experiment. Open from the repository root or `notebooks/`.
See [the research report](../docs/report.md) for interpretation and limitations.
The data cache is local and excluded from Git; run `python main.py` to create it.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')
sys.path.insert(0, str(ROOT))

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data import CHAR_NAMES

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline


## 1. Load cached data

In [ ]:
CACHE = ROOT / 'data/raw/data_cache_v4.pkl'
if not CACHE.exists():
    raise FileNotFoundError(f'{CACHE} does not exist. Run python main.py from the repository root first.')
with CACHE.open('rb') as f:
    data = pickle.load(f)  # Only load trusted local caches.
if data.get('cache_version') != 4:
    raise ValueError('Use the current 19-feature cache.')
splits, returns, chars = data['splits'], data['returns'], data['chars']
dates, tickers = data['dates'], data['tickers']
print(f'Full sample: {len(returns)} months × {returns.shape[1]} stocks')
print(f'Date range: {dates[0].date()} → {dates[-1].date()}')


## 2. Return distribution

In [ ]:
ret_vals = returns.values.flatten()
ret_vals = ret_vals[~np.isnan(ret_vals)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(ret_vals, bins=200, range=(-0.5, 0.5), color='steelblue', alpha=0.7)
axes[0].set_title('Monthly Excess Return Distribution')
axes[0].set_xlabel('Monthly Return')
axes[0].set_ylabel('Frequency')

# Coverage: % stocks available per month
coverage = returns.notna().mean(axis=1) * 100
axes[1].plot(returns.index, coverage, lw=0.8)
axes[1].set_title('Stock Coverage Over Time (% of universe)')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('% Stocks Available')

plt.tight_layout()
plt.show()

print(f'Mean excess return: {ret_vals.mean():.4f}')
print(f'Std  excess return: {ret_vals.std():.4f}')
print(f'Skewness:           {pd.Series(ret_vals).skew():.3f}')
print(f'Kurtosis:           {pd.Series(ret_vals).kurtosis():.3f}')

## 3. Characteristics distributions

In [ ]:
char_names = data.get('characteristic_names', CHAR_NAMES)
fig, axes = plt.subplots(3, 7, figsize=(20, 9))
for p, (ax, name) in enumerate(zip(axes.flat, char_names)):
    vals = chars[:, :, p].ravel()
    ax.hist(vals[np.isfinite(vals)], bins=50, color='darkorange', alpha=0.7)
    ax.set_title(name)
for ax in list(axes.flat)[len(char_names):]:
    ax.set_visible(False)
plt.suptitle('All 19 characteristics, cross-sectionally ranked to [-1, 1]')
plt.tight_layout()
plt.show()


## 4. PCA variance explained

In [ ]:
from sklearn.decomposition import PCA

train_ret = splits['train']['returns'].values
counts = np.isfinite(train_ret).sum(axis=0)
col_means = np.divide(np.nansum(train_ret, axis=0), counts,
                      out=np.zeros(train_ret.shape[1]), where=counts > 0)
filled    = np.where(np.isnan(train_ret), col_means[None,:], train_ret)

pca_full = PCA(n_components=20).fit(filled)
var_exp  = pca_full.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(var_exp)+1), var_exp * 100, color='steelblue')
ax.plot(range(1, len(var_exp)+1), np.cumsum(var_exp)*100,
        'ro-', markersize=4, label='Cumulative')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('PCA Scree Plot (Training Data)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Load saved summary table

In [ ]:
# Choose the run explicitly; data/results never silently switch experiments.
RUN_DIR = ROOT / 'results/full-2026-09-22-clean'
summary = pd.read_csv(RUN_DIR / 'summary_table.csv')
print(f'Results: {RUN_DIR.name}')
summary


In [ ]:
# Heatmap: Total R2 by Model and K
df_num = summary.copy()
df_num['Total_R2'] = pd.to_numeric(df_num['Total_R2'], errors='coerce')
df_num['Sharpe']   = pd.to_numeric(df_num['Sharpe'],   errors='coerce')

pivot = df_num.pivot_table(index='Model', columns='K',
                            values='Total_R2', aggfunc='mean')

fig, ax = plt.subplots(figsize=(7, 3))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn', ax=ax)
ax.set_title(f'Total R² — run {RUN_DIR.name} (reconstruction, not forecasts)')
plt.tight_layout()
plt.show()